<a href="https://colab.research.google.com/github/NIKUNJ-PROGRAMMER/PRE-PLACEMENT/blob/main/Walmert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing required libraries

In [13]:
# Data manipulation and calculations
import pandas as pd
import numpy as np

# Machine Learning and Evaluation metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Tool for saving the final model
import joblib

# Ignore warning messages for a cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


Loading and Inspecting the Dataset

In [14]:
# Load the dataset (ensure the file is uploaded to your Colab environment)
df = pd.read_csv('Walmart (1).csv')

# Display the first 5 rows to confirm it loaded correctly
print("First 5 rows of the dataset:")
display(df.head())

# Check for missing values and data types
print("\nDataset Information:")
display(df.info())
print("\nMissing Values Count:")
display(df.isnull().sum())

First 5 rows of the dataset:


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


None


Missing Values Count:


,0
Store,0
Date,0
Weekly_Sales,0
Holiday_Flag,0
Temperature,0
Fuel_Price,0
CPI,0
Unemployment,0


Data Pre-processing & Feature Engineering

In [15]:
# Convert the 'Date' column to a standard pandas datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Extract temporal features for seasonality
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

print("Temporal features (Week, Month, Year) successfully extracted!")
display(df[['Date', 'Week', 'Month', 'Year']].head())

Temporal features (Week, Month, Year) successfully extracted!


,Date,Week,Month,Year
0,2010-02-05,5,2,2010
1,2010-02-12,6,2,2010
2,2010-02-19,7,2,2010
3,2010-02-26,8,2,2010
4,2010-03-05,9,3,2010


Defining Features and train-test split

In [16]:
# Define independent variables (X) and the target variable (y)
# We drop 'Weekly_Sales' (target) and 'Date' (already engineered)
X = df.drop(columns=['Weekly_Sales', 'Date'])
y = df['Weekly_Sales']

# Split 80% of the data for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (5148, 9)
Testing data shape: (1287, 9)


Iniatializing and Training model

In [17]:
# Initialize the Random Forest model with 100 decision trees
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train (fit) the model on the training data
rf_model.fit(X_train, y_train)

print("Random Forest model has been successfully trained!")

Random Forest model has been successfully trained!


Evaluating Model Performance

In [18]:
# Make predictions on the testing data
y_pred = rf_model.predict(X_test)

# Calculate performance metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- MODEL EVALUATION METRICS ---")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-Squared (R2) Score: {r2:.4f} ({(r2*100):.2f}%)")

--- MODEL EVALUATION METRICS ---
Root Mean Squared Error (RMSE): 114335.11
R-Squared (R2) Score: 0.9594 (95.94%)


Extracting Business Insights

In [19]:
# Extract feature importances from the trained model
importances = rf_model.feature_importances_

# Map importances to their respective column names and sort them
feature_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
feature_imp_percentage = (feature_imp * 100).round(2)

print("--- BUSINESS INSIGHTS: FEATURE IMPORTANCE (%) ---")
print(feature_imp_percentage)

--- BUSINESS INSIGHTS: FEATURE IMPORTANCE (%) ---
Store           66.33
CPI             15.46
Unemployment    10.37
Week             5.00
Temperature      1.28
Fuel_Price       1.03
Month            0.29
Holiday_Flag     0.15
Year             0.09
dtype: float64


Forecasting Sales for the Next 12 weeks

In [20]:
# Train a final model on the full dataset for maximum accuracy
rf_final = RandomForestRegressor(n_estimators=100, random_state=42)
rf_final.fit(X, y)

# Generate future dates (12 weeks from the last date in the dataset)
last_date = df['Date'].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=7), periods=12, freq='W-FRI')

future_data = []
# Loop through every unique store to generate individual forecasts
for store in df['Store'].unique():
    # Capture the most recent economic data for this specific store
    store_last = df[df['Store'] == store].sort_values('Date').iloc[-1]

    for date in future_dates:
        future_data.append({
            'Store': store,
            'Holiday_Flag': 0, # Baseline assumption of non-holidays
            'Temperature': store_last['Temperature'],
            'Fuel_Price': store_last['Fuel_Price'],
            'CPI': store_last['CPI'],
            'Unemployment': store_last['Unemployment'],
            'Week': date.isocalendar().week,
            'Month': date.month,
            'Year': date.year
        })

# Convert to DataFrame and predict
future_df = pd.DataFrame(future_data)
future_df['Forecasted_Weekly_Sales'] = rf_final.predict(future_df)

# Export the predictions to a CSV file
future_df.to_csv('walmart_12_week_forecast.csv', index=False)
print("12-Week Forecast successfully generated and saved as 'walmart_12_week_forecast.csv'!")
display(future_df.head(12)) # Display the forecast for Store 1

12-Week Forecast successfully generated and saved as 'walmart_12_week_forecast.csv'!


,Store,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Week,Month,Year,Forecasted_Weekly_Sales
0,1,0,69.16,3.506,223.444251,6.573,44,11,2012,1.513961e+06
1,1,0,69.16,3.506,223.444251,6.573,45,11,2012,1.513961e+06
2,1,0,69.16,3.506,223.444251,6.573,46,11,2012,1.513961e+06
3,1,0,69.16,3.506,223.444251,6.573,47,11,2012,1.863658e+06
4,1,0,69.16,3.506,223.444251,6.573,48,11,2012,1.863658e+06
5,1,0,69.16,3.506,223.444251,6.573,49,12,2012,1.906713e+06
6,1,0,69.16,3.506,223.444251,6.573,50,12,2012,1.966890e+06
7,1,0,69.16,3.506,223.444251,6.573,51,12,2012,2.123784e+06
8,1,0,69.16,3.506,223.444251,6.573,52,12,2012,1.909983e+06
9,1,0,69.16,3.506,223.444251,6.573,1,1,2013,1.547224e+06


Saving Final Model

In [21]:
# Save the model to a file
joblib.dump(rf_model, 'walmart_sales_model.pkl')
print("Model saved successfully as 'walmart_sales_model.pkl'. You can now download it from your file explorer.")

Model saved successfully as 'walmart_sales_model.pkl'. You can now download it from your file explorer.
